In [ ]:
print("H")

In [ ]:
# Shared setup — imports, constants, paths
import os, json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from facenet_pytorch import MTCNN
from scipy.fft import dctn
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve,
)
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

# Backend-compatible constants
KYC_MAX_VIDEO_FRAMES = 5
KYC_FRAME_SIZE       = 224

ROOT            = Path("../../")
DATA_DIR        = ROOT / "data" / "kyc"
CROPS_DIR       = DATA_DIR / "crops"
WEIGHTS_DIR     = ROOT / "backend" / "weights"
PRECOMPUTED_DIR = ROOT / "backend" / "precomputed"
RESULTS_DIR     = Path("results")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
PRECOMPUTED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED   = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Device:", DEVICE)

In [ ]:
# Build video-level 80/10/10 split from pre-extracted face crops
# Crop filenames: <source>_<videostem>_f<N>.jpg
# The split is at video level so no video leaks across splits.

def build_split(label):
    crops = list((CROPS_DIR / label).glob("*.jpg"))
    groups = {}
    for p in crops:
        key = p.stem.rsplit("_", 1)[0]
        groups.setdefault(key, []).append(p)
    videos = list(groups.keys())
    train_v, temp_v = train_test_split(videos, test_size=0.2, random_state=SEED)
    val_v,   test_v = train_test_split(temp_v, test_size=0.5, random_state=SEED)
    return {s: [p for v in vs for p in groups[v]]
            for s, vs in [("train",train_v),("val",val_v),("test",test_v)]}

real_splits = build_split("real")
fake_splits = build_split("fake")

split_dfs = {}
for split in ("train", "val", "test"):
    rows = [(str(p), 0) for p in real_splits[split]] + \
           [(str(p), 1) for p in fake_splits[split]]
    split_dfs[split] = pd.DataFrame(rows, columns=["path", "label"])
    n  = len(split_dfs[split])
    nr = (split_dfs[split].label == 0).sum()
    nf = (split_dfs[split].label == 1).sum()
    print(f"{split:5s}: {n} samples  (real={nr}, fake={nf})")

test_paths = split_dfs["test"]["path"].values

In [ ]:
# ImageNet normalization — same as backend inference.py
IMAGENET_TRANSFORM = transforms.Compose([
    transforms.Resize((KYC_FRAME_SIZE, KYC_FRAME_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
AUGMENT_TRANSFORM = transforms.Compose([
    transforms.Resize((KYC_FRAME_SIZE, KYC_FRAME_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

class FaceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform or IMAGENET_TRANSFORM
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.transform(Image.open(row["path"]).convert("RGB")), int(row["label"])

def make_loaders(batch_size=32):
    train_dl = DataLoader(FaceDataset(split_dfs["train"], AUGMENT_TRANSFORM),
                          batch_size=batch_size, shuffle=True,  num_workers=2)
    val_dl   = DataLoader(FaceDataset(split_dfs["val"]),
                          batch_size=batch_size, shuffle=False, num_workers=2)
    test_dl  = DataLoader(FaceDataset(split_dfs["test"]),
                          batch_size=batch_size, shuffle=False, num_workers=2)
    return train_dl, val_dl, test_dl

In [ ]:
# FrequencyCNN — matches backend/models/kyc/frequency_cnn.py exactly
class FrequencyCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),          nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),         nn.ReLU(), nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))

In [ ]:
# DCT feature extractor — matches backend frequency_analyzer.extract_dct_features exactly
def extract_dct_features(face_image, size=224):
    gray    = face_image.convert("L").resize((size, size))
    arr     = np.array(gray, dtype=np.float32) / 255.0
    dct     = dctn(arr, norm="ortho")
    log_mag = np.log1p(np.abs(dct))
    log_mag = (log_mag - log_mag.min()) / (log_mag.max() - log_mag.min() + 1e-8)
    return log_mag[np.newaxis, :, :]   # (1, H, W)

class DCTDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        dct = extract_dct_features(Image.open(row["path"]).convert("RGB"))
        return torch.tensor(dct, dtype=torch.float), int(row["label"])

def make_dct_loaders(batch_size=32):
    train_dl = DataLoader(DCTDataset(split_dfs["train"]), batch_size=batch_size, shuffle=True,  num_workers=2)
    val_dl   = DataLoader(DCTDataset(split_dfs["val"]),   batch_size=batch_size, shuffle=False, num_workers=2)
    test_dl  = DataLoader(DCTDataset(split_dfs["test"]),  batch_size=batch_size, shuffle=False, num_workers=2)
    return train_dl, val_dl, test_dl

In [ ]:
# Training utilities shared by all RGB experiments

def train_one_epoch(model, loader, optimizer, criterion):
    model.train(); total_loss = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)

def evaluate_loader(model, loader):
    model.eval(); all_labels, all_probs = [], []
    with torch.no_grad():
        for x, y in loader:
            prob = F.softmax(model(x.to(DEVICE)), dim=1)[:,1].cpu().numpy()
            all_probs.extend(prob); all_labels.extend(y.numpy())
    return np.array(all_labels), np.array(all_probs)

def train_model(model, train_dl, val_dl, optimizer, scheduler,
                epochs=20, save_path=None, patience=5):
    criterion = nn.CrossEntropyLoss()
    best_auc, wait = 0.0, 0
    history = {"train_loss": [], "val_auc": []}
    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(model, train_dl, optimizer, criterion)
        labels, probs = evaluate_loader(model, val_dl)
        val_auc = roc_auc_score(labels, probs)
        history["train_loss"].append(loss)
        history["val_auc"].append(val_auc)
        if scheduler: scheduler.step()
        print(f"Epoch {epoch:02d}/{epochs}  loss={loss:.4f}  val_auc={val_auc:.4f}")
        if val_auc > best_auc:
            best_auc, wait = val_auc, 0
            if save_path:
                torch.save(model.state_dict(), save_path)
                print(f"  -> Checkpoint saved (auc={best_auc:.4f})")
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch}"); break
    return history

In [ ]:
# Standard evaluation function used by every experiment

def evaluate(y_true, y_pred_prob, threshold=0.5, title="Model"):
    y_pred = (np.array(y_pred_prob) >= threshold).astype(int)
    y_true = np.array(y_true)
    metrics = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1":        f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC":   roc_auc_score(y_true, y_pred_prob),
        "PR-AUC":    average_precision_score(y_true, y_pred_prob),
    }
    print(f"\n--- {title} ---")
    for k, v in metrics.items(): print(f"  {k:12s}: {v:.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(title)
    cm = confusion_matrix(y_true, y_pred)
    axes[0].imshow(cm, cmap="Blues")
    axes[0].set_title("Confusion Matrix")
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, cm[i,j], ha="center", va="center", fontsize=14)
    axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
    axes[0].set_xticklabels(["Real","Fake"]); axes[0].set_yticklabels(["Real","Fake"])
    fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
    axes[1].plot(fpr, tpr, lw=2, label=f"AUC={metrics['ROC-AUC']:.3f}")
    axes[1].plot([0,1],[0,1],"--",color="grey")
    axes[1].set_title("ROC Curve"); axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
    axes[1].legend()
    prec, rec, _ = precision_recall_curve(y_true, y_pred_prob)
    axes[2].plot(rec, prec, lw=2, label=f"AP={metrics['PR-AUC']:.3f}")
    axes[2].set_title("PR Curve"); axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision")
    axes[2].legend()
    plt.tight_layout(); plt.show()
    return metrics

def threshold_analysis(y_true, y_pred_prob, title="Threshold Analysis"):
    thresholds = np.linspace(0.1, 0.9, 50)
    f1s, precs, recs = [], [], []
    for t in thresholds:
        yp = (np.array(y_pred_prob) >= t).astype(int)
        f1s.append(f1_score(y_true, yp, zero_division=0))
        precs.append(precision_score(y_true, yp, zero_division=0))
        recs.append(recall_score(y_true, yp, zero_division=0))
    plt.figure(figsize=(8,4))
    plt.plot(thresholds, f1s,   label="F1")
    plt.plot(thresholds, precs, label="Precision")
    plt.plot(thresholds, recs,  label="Recall")
    plt.axvline(0.50, color="red",    linestyle="--", label="suspicious (0.50)")
    plt.axvline(0.75, color="orange", linestyle="--", label="high_risk (0.75)")
    plt.title(title); plt.xlabel("Threshold"); plt.legend()
    plt.tight_layout(); plt.show()

In [ ]:
# Helper: display a grid of face crops with their model score
def show_examples(indices, probs, title, n=6):
    indices = list(indices[:n])
    if not indices: print(f"{title}: none"); return
    fig, axes = plt.subplots(1, len(indices), figsize=(3*len(indices), 3))
    if len(indices) == 1: axes = [axes]
    for ax, i in zip(axes, indices):
        ax.imshow(Image.open(test_paths[i])); ax.axis("off")
        ax.set_title(f"p={probs[i]:.2f}", fontsize=8)
    fig.suptitle(title); plt.tight_layout(); plt.show()

In [ ]:
# Experiment 2.6 — Cross-Dataset Robustness + Error Analysis
# Question: Does the model generalize across FaceForensics++ and Celeb-DF v2?
# Additional: deep-dive into ensemble errors and model disagreements.

# Load all three production models from checkpoints
EFFNET_CKPT = WEIGHTS_DIR / "efficientnet_best.pt"
VIT_CKPT    = WEIGHTS_DIR / "vit_best.pt"
FREQ_CKPT   = WEIGHTS_DIR / "frequency_cnn_best.pt"

effnet = timm.create_model("efficientnet_b4", pretrained=False, num_classes=2)
effnet.load_state_dict(torch.load(EFFNET_CKPT, map_location=DEVICE))
effnet.eval().to(DEVICE)

vit = timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=2)
vit.load_state_dict(torch.load(VIT_CKPT, map_location=DEVICE))
vit.eval().to(DEVICE)

freq_cnn = FrequencyCNN().to(DEVICE)
freq_cnn.load_state_dict(torch.load(FREQ_CKPT, map_location=DEVICE))
freq_cnn.eval()

print("All three models loaded.")

In [ ]:
# Per-source test loaders
from torch.utils.data import DataLoader

class DCTDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        dct = extract_dct_features(Image.open(row["path"]).convert("RGB"))
        return torch.tensor(dct, dtype=torch.float), int(row["label"])

def get_source_loader(source_name, dct=False, batch_size=32):
    mask = split_dfs["test"]["path"].str.contains(source_name)
    sub  = split_dfs["test"][mask]
    if len(sub) == 0: print(f"No test samples for {source_name}"); return None
    ds = DCTDataset(sub) if dct else FaceDataset(sub)
    return DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
# Cross-dataset robustness evaluation
robustness_rows = []
sources = {"FaceForensics++": "faceforensics", "Celeb-DF v2": "celebdf"}

for display_name, source_key in sources.items():
    rgb_dl = get_source_loader(source_key)
    dct_dl = get_source_loader(source_key, dct=True)
    if rgb_dl is None: continue

    y_true, p_e = evaluate_loader(effnet,   rgb_dl)
    _,       p_v = evaluate_loader(vit,       rgb_dl)
    _,       p_f = evaluate_loader(freq_cnn, dct_dl)
    p_ens = (p_e + p_v + p_f) / 3

    for name, probs in [("EfficientNet-B4",p_e),("ViT-B/16",p_v),
                         ("FrequencyCNN",p_f),("Ensemble",p_ens)]:
        from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
        robustness_rows.append({
            "Model": name, "Dataset": display_name,
            "ROC-AUC": round(roc_auc_score(y_true, probs), 4),
            "PR-AUC":  round(average_precision_score(y_true, probs), 4),
            "F1":      round(f1_score(y_true,(probs>=0.5).astype(int),zero_division=0), 4),
        })

robustness_df = pd.DataFrame(robustness_rows)
pivot = robustness_df.pivot(index="Model", columns="Dataset")
print(pivot.to_string())

In [ ]:
# Robustness heatmap
for metric in ["ROC-AUC","PR-AUC","F1"]:
    sub = robustness_df.pivot(index="Model", columns="Dataset", values=metric)
    fig, ax = plt.subplots(figsize=(5,4))
    im = ax.imshow(sub.values, cmap="RdYlGn", vmin=0.5, vmax=1.0)
    ax.set_xticks(range(len(sub.columns))); ax.set_xticklabels(sub.columns, rotation=20, ha="right")
    ax.set_yticks(range(len(sub.index)));  ax.set_yticklabels(sub.index)
    for i in range(len(sub.index)):
        for j in range(len(sub.columns)):
            ax.text(j, i, f"{sub.values[i,j]:.3f}", ha="center", va="center", fontsize=10)
    plt.colorbar(im, ax=ax); ax.set_title(f"Cross-Dataset {metric}")
    plt.tight_layout(); plt.show()

In [ ]:
# Error analysis — full test set ensemble
probs_eff   = np.load(RESULTS_DIR / "exp2.2_probs.npy")
probs_vit   = np.load(RESULTS_DIR / "exp2.3_probs.npy")
probs_freq  = np.load(RESULTS_DIR / "exp2.4_probs.npy")
labels_eff  = np.load(RESULTS_DIR / "exp2.2_labels.npy")
probs_ensemble = (probs_eff + probs_vit + probs_freq) / 3

preds_ens = (probs_ensemble >= 0.5).astype(int)
fp_ens = np.where((labels_eff==0) & (preds_ens==1))[0]
fn_ens = np.where((labels_eff==1) & (preds_ens==0))[0]
print(f"Ensemble false positives: {len(fp_ens)}")
print(f"Ensemble false negatives: {len(fn_ens)}")

In [ ]:
# Show false positives (Real predicted as Fake)
test_paths = split_dfs["test"]["path"].values
show_examples(fp_ens, probs_ensemble, "Ensemble FP — Real predicted as Fake")

In [ ]:
# Show false negatives (Fake predicted as Real)
show_examples(fn_ens, probs_ensemble, "Ensemble FN — Fake predicted as Real")

In [ ]:
# Show high-disagreement samples (EfficientNet vs ViT)
disagreement = np.abs(probs_eff - probs_vit)
top_idx = np.argsort(disagreement)[::-1][:6]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.flatten()
for ax, i in zip(axes, top_idx):
    ax.imshow(Image.open(test_paths[i])); ax.axis("off")
    true_label = "Fake" if labels_eff[i] else "Real"
    ax.set_title(
        f"True={true_label}\nEff={probs_eff[i]:.2f} ViT={probs_vit[i]:.2f} Freq={probs_freq[i]:.2f}",
        fontsize=7)
fig.suptitle("Top Model Disagreements (EfficientNet vs ViT)")
plt.tight_layout(); plt.show()

In [ ]:
# Save robustness data for precompute notebook
robustness_df.to_csv(RESULTS_DIR / "exp2.6_robustness.csv", index=False)
print("exp2.6 done")